<a href="https://colab.research.google.com/github/Aletz44/Actividad-5_ABG/blob/main/fuentes/Validacion_Restauracion_MLflow_ABG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# TAREA 5
# Restauración de MLFLOW
# 1. Carga del respaldo original

from google.colab import files
from pathlib import Path

print("=" * 65)
print("1. CARGA DEL RESPALDO DE MLFLOW")
print("=" * 65)

archivos = files.upload()

ruta_zip = Path("/content/mlflow_rama_final.zip")

assert ruta_zip.exists(), (
    "No se encontró mlflow_rama_final.zip."
)

print("\nArchivo:", ruta_zip.name)
print("Tamaño:", ruta_zip.stat().st_size, "bytes")
print("Respaldo cargado correctamente.")

1. CARGA DEL RESPALDO DE MLFLOW


Saving mlflow_rama_final.zip to mlflow_rama_final.zip

Archivo: mlflow_rama_final.zip
Tamaño: 1766871 bytes
Respaldo cargado correctamente.


In [2]:
# TAREA 5
# Restauración de MLFLOW
# 2. Extracción y verificación del respaldo

import zipfile
import sqlite3
import shutil
import json
from pathlib import Path

# Verificación del archivo ZIP
print("=" * 65)
print("1. VERIFICACIÓN DEL RESPALDO")
print("=" * 65)

ruta_zip = Path("/content/mlflow_rama_final.zip")
ruta_restauracion = Path("/content/mlflow_restaurado")

assert ruta_zip.exists(), (
    "No se encontró el archivo de respaldo."
)

with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

    assert archivo_zip.testzip() is None, (
        "El archivo ZIP contiene errores."
    )

    nombres_archivos = archivo_zip.namelist()

print("Archivo:", ruta_zip.name)
print("Archivos incluidos:", len(nombres_archivos))
print("Integridad ZIP: CORRECTA")

# Extracción controlada
print("\n" + "=" * 65)
print("2. EXTRACCIÓN DEL RESPALDO")
print("=" * 65)

# Evitar sobrescribir una restauración anterior.
assert not ruta_restauracion.exists(), (
    "La carpeta de restauración ya existe."
)

ruta_restauracion.mkdir(parents=True)

with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

    for miembro in archivo_zip.infolist():

        destino = (
            ruta_restauracion / miembro.filename
        ).resolve()

        if not destino.is_relative_to(
            ruta_restauracion.resolve()
        ):
            raise ValueError(
                "Ruta insegura detectada en el ZIP."
            )

        if miembro.is_dir():
            destino.mkdir(parents=True, exist_ok=True)
        else:
            destino.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            with archivo_zip.open(miembro) as origen:
                with open(destino, "wb") as salida:
                    shutil.copyfileobj(origen, salida)

print("Extracción: COMPLETADA")
print("Carpeta:", ruta_restauracion)

# Integridad de SQLITE
print("\n" + "=" * 65)
print("3. VERIFICACIÓN DE LA BASE DE DATOS")
print("=" * 65)

ruta_db = ruta_restauracion / "mlflow.db"

assert ruta_db.exists(), (
    "No se encontró mlflow.db."
)

conexion = sqlite3.connect(
    f"file:{ruta_db}?mode=ro",
    uri=True
)

try:

    integridad = conexion.execute(
        "PRAGMA integrity_check"
    ).fetchone()[0]

    assert integridad == "ok"

    tablas = [
        fila[0]
        for fila in conexion.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type='table'"
        ).fetchall()
    ]

    assert "experiments" in tablas
    assert "runs" in tablas

    print("Integridad SQLite: CORRECTA")
    print("Tablas encontradas:", len(tablas))

finally:
    conexion.close()

# Verificación del manifiesto
print("\n" + "=" * 65)
print("4. VERIFICACIÓN DEL MANIFIESTO")
print("=" * 65)

ruta_manifiesto = (
    ruta_restauracion / "manifiesto.json"
)

assert ruta_manifiesto.exists(), (
    "No se encontró manifiesto.json."
)

with open(
    ruta_manifiesto,
    "r",
    encoding="utf-8"
) as archivo:

    manifiesto = json.load(archivo)

assert manifiesto["experimento"] == (
    "RAMA_PM25_Actividad5"
)

print("Experimento:", manifiesto["experimento"])
print("MLflow:", manifiesto["mlflow_version"])
print("Dataset SHA-256:", manifiesto["dataset_sha256"])
print("Manifiesto: CORRECTO")

# Inventario de archivos
print("\n" + "=" * 65)
print("5. INVENTARIO DE ARCHIVOS RESTAURADOS")
print("=" * 65)

archivos_restaurados = [
    archivo
    for archivo in ruta_restauracion.rglob("*")
    if archivo.is_file()
]

print("Archivos restaurados:", len(archivos_restaurados))

for elemento in sorted(
    ruta_restauracion.iterdir()
):
    print("-", elemento.name)

# Resumen final
print("\n" + "=" * 65)
print("6. RESULTADO DE LA RESTAURACIÓN ESTRUCTURAL")
print("=" * 65)

print("Archivo ZIP: VERIFICADO")
print("Extracción: COMPLETADA")
print("Base de datos SQLite: CORRECTA")
print("Manifiesto: CORRECTO")
print("Restauración estructural: SATISFACTORIA")

print("\nPendiente:")
print("Verificar las rutas de almacenamiento y")
print("recuperar los modelos desde la copia restaurada.")


1. VERIFICACIÓN DEL RESPALDO
Archivo: mlflow_rama_final.zip
Archivos incluidos: 27
Integridad ZIP: CORRECTA

2. EXTRACCIÓN DEL RESPALDO
Extracción: COMPLETADA
Carpeta: /content/mlflow_restaurado

3. VERIFICACIÓN DE LA BASE DE DATOS
Integridad SQLite: CORRECTA
Tablas encontradas: 65

4. VERIFICACIÓN DEL MANIFIESTO
Experimento: RAMA_PM25_Actividad5
MLflow: 3.17.1
Dataset SHA-256: 93c76aa30597174fce7bbd49092aafdd586bd56786415091bb3e79177dab1d8b
Manifiesto: CORRECTO

5. INVENTARIO DE ARCHIVOS RESTAURADOS
Archivos restaurados: 27
- artefactos
- manifiesto.json
- mlflow.db

6. RESULTADO DE LA RESTAURACIÓN ESTRUCTURAL
Archivo ZIP: VERIFICADO
Extracción: COMPLETADA
Base de datos SQLite: CORRECTA
Manifiesto: CORRECTO
Restauración estructural: SATISFACTORIA

Pendiente:
Verificar las rutas de almacenamiento y
recuperar los modelos desde la copia restaurada.


In [3]:
# TAREA 5
# Restauración de MLFLOW
# 3. Diagnóstico de rutas y modelos restaurados

import sys
import subprocess
import sqlite3
from pathlib import Path

# Verificación del entorno
print("=" * 65)
print("1. VERIFICACIÓN DEL ENTORNO")
print("=" * 65)

ruta_restauracion = Path("/content/mlflow_restaurado")
ruta_db = ruta_restauracion / "mlflow.db"
ruta_artefactos = ruta_restauracion / "artefactos"

assert ruta_db.exists(), "No se encontró la base restaurada."

try:
    import mlflow
except ImportError:
    print("Instalando MLflow 3.17.1...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "mlflow==3.17.1"
        ],
        check=True
    )
    import mlflow

from mlflow.tracking import MlflowClient

print("MLflow:", mlflow.__version__)
print("Base restaurada:", ruta_db)
print("Carpeta de artefactos:", ruta_artefactos.exists())

# Consulta del experimento restaurado
print("\n" + "=" * 65)
print("2. CONSULTA DEL EXPERIMENTO")
print("=" * 65)

uri_anterior = mlflow.get_tracking_uri()

mlflow.set_tracking_uri(
    f"sqlite:///{ruta_db.resolve()}"
)

cliente = MlflowClient()

experimento = mlflow.get_experiment_by_name(
    "RAMA_PM25_Actividad5"
)

assert experimento is not None, (
    "El experimento no aparece en la base restaurada."
)

print("Experimento:", experimento.name)
print("ID:", experimento.experiment_id)
print("Ubicación de artefactos:", experimento.artifact_location)

# Consulta de modelos registrados
print("\n" + "=" * 65)
print("3. MODELOS REGISTRADOS")
print("=" * 65)

modelos = cliente.search_logged_models(
    experiment_ids=[str(experimento.experiment_id)]
)

for modelo in modelos:

    print("\nModelo:", modelo.name)
    print("Model ID:", modelo.model_id)
    print("Estado:", modelo.status)
    print("Run origen:", modelo.source_run_id)

    # Consultar los metadatos disponibles.
    for atributo in [
        "artifact_location",
        "source",
        "model_uri"
    ]:
        valor = getattr(modelo, atributo, None)

        if valor is not None:
            print(atributo + ":", valor)

# Inventario de archivos de modelos
print("\n" + "=" * 65)
print("4. ARCHIVOS DE MODELOS RESTAURADOS")
print("=" * 65)

archivos_modelo = [
    archivo
    for archivo in ruta_restauracion.rglob("*")
    if archivo.is_file()
    and archivo.name in [
        "MLmodel",
        "model.pkl",
        "conda.yaml",
        "python_env.yaml"
    ]
]

print("Archivos de modelo encontrados:", len(archivos_modelo))

for archivo in archivos_modelo:
    print("-", archivo.relative_to(ruta_restauracion))

# Consulta de rutas en SQLITE
print("\n" + "=" * 65)
print("5. REFERENCIAS DE ALMACENAMIENTO")
print("=" * 65)

conexion = sqlite3.connect(
    f"file:{ruta_db}?mode=ro",
    uri=True
)

try:
    tablas = [
        fila[0]
        for fila in conexion.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type='table'"
        ).fetchall()
    ]

    for tabla in ["experiments", "runs", "logged_models"]:

        if tabla not in tablas:
            continue

        columnas = [
            fila[1]
            for fila in conexion.execute(
                f"PRAGMA table_info({tabla})"
            ).fetchall()
        ]

        columnas_ruta = [
            columna
            for columna in columnas
            if "artifact" in columna.lower()
            or "source" in columna.lower()
        ]

        print("\nTabla:", tabla)
        print("Columnas relacionadas:", columnas_ruta)

        for columna in columnas_ruta:

            filas = conexion.execute(
                f'SELECT DISTINCT "{columna}" '
                f'FROM "{tabla}" '
                f'WHERE "{columna}" IS NOT NULL '
                f'LIMIT 5'
            ).fetchall()

            for fila in filas:
                print("-", fila[0])

finally:
    conexion.close()

# Resultado del diagnóstico
print("\n" + "=" * 65)
print("6. RESUMEN")
print("=" * 65)

print("Base restaurada: CONSULTADA")
print("Experimento: ENCONTRADO")
print("Modelos registrados:", len(modelos))
print("Archivos de modelo identificados:", len(archivos_modelo))
print("Diagnóstico de rutas: COMPLETADO")

print("\nTodavía no se han cargado los modelos.")
print("No se modificó la base de datos restaurada.")

1. VERIFICACIÓN DEL ENTORNO
Instalando MLflow 3.17.1...
MLflow: 3.17.1
Base restaurada: /content/mlflow_restaurado/mlflow.db
Carpeta de artefactos: True

2. CONSULTA DEL EXPERIMENTO
Experimento: RAMA_PM25_Actividad5
ID: 1
Ubicación de artefactos: file:///content/mlflow_rama/artefactos

3. MODELOS REGISTRADOS

Modelo: random_forest_pm25
Model ID: m-8de0647dad4c468d8e1308d0f261f528
Estado: READY
Run origen: 3c164f2beea644eba2f4068cb3b7beb6
artifact_location: file:///content/mlflow_rama/artefactos/models/m-8de0647dad4c468d8e1308d0f261f528/artifacts
model_uri: models:/m-8de0647dad4c468d8e1308d0f261f528

Modelo: random_forest_pm25
Model ID: m-6b8af92063aa46859d33809fac8403d3
Estado: FAILED
Run origen: 764e326660d64a458fc3c3c6ec8f6c3d
artifact_location: file:///content/mlflow_rama/artefactos/models/m-6b8af92063aa46859d33809fac8403d3/artifacts
model_uri: models:/m-6b8af92063aa46859d33809fac8403d3

Modelo: ridge_pm25
Model ID: m-c6696e03f8294f8aa3075a56415f6b41
Estado: READY
Run origen: 73339b

In [6]:
# TAREA 5
# Restauración de MLFLOW
# 4. Corrección de rutas y recuperación de modelos

import sys
import subprocess
import sqlite3
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import yaml

from mlflow.tracking import MlflowClient

# Configuración y verificación inicial
print("=" * 65)
print("1. CONFIGURACIÓN DE LA RESTAURACIÓN")
print("=" * 65)

ruta_restauracion = Path("/content/mlflow_restaurado")
ruta_db = ruta_restauracion / "mlflow.db"
ruta_artefactos = ruta_restauracion / "artefactos"

ruta_antigua = "file:///content/mlflow_rama/artefactos"
ruta_nueva = ruta_artefactos.resolve().as_uri()

assert ruta_db.exists(), "No se encontró la base restaurada."
assert ruta_artefactos.exists(), (
    "No se encontró la carpeta de artefactos."
)

assert mlflow.active_run() is None, (
    "Existe una ejecución activa en MLflow."
)

print("MLflow:", mlflow.__version__)
print("Base restaurada:", ruta_db)
print("Artefactos:", ruta_artefactos)
print("Ruta esperada:", ruta_nueva)

# Verificación de dependencias de serialización
print("\n" + "=" * 65)
print("2. VERIFICACIÓN DE DEPENDENCIAS")
print("=" * 65)

# Ridge utiliza skops y Random Forest cloudpickle.
# Se comprueba su disponibilidad antes de cargar modelos.

try:
    import skops.io
    print("skops: DISPONIBLE")

except ImportError:

    print("Instalando skops...")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "skops"
        ],
        check=True
    )

    import skops.io

try:
    import cloudpickle
    print("cloudpickle:", cloudpickle.__version__)

except ImportError:

    raise RuntimeError(
        "No se encontró cloudpickle."
    )

# Verificación y corrección de rutas
print("\n" + "=" * 65)
print("3. VERIFICACIÓN DE REFERENCIAS SQLITE")
print("=" * 65)

# Se conserva una copia antes de cualquier modificación.
ruta_seguridad = (
    ruta_restauracion / "mlflow_antes_rutas.db"
)

if not ruta_seguridad.exists():

    shutil.copy2(
        ruta_db,
        ruta_seguridad
    )

    print("Copia de seguridad: GENERADA")

else:

    print("Copia de seguridad: EXISTENTE")

conexion = sqlite3.connect(str(ruta_db))

tablas_columnas = {
    "experiments": ["artifact_location"],
    "runs": ["artifact_uri"],
    "logged_models": ["artifact_location"]
}

total_actualizaciones = 0

try:

    conexion.execute("BEGIN IMMEDIATE")

    for tabla, columnas in tablas_columnas.items():

        for columna in columnas:

            cursor = conexion.execute(
                f"""
                UPDATE {tabla}
                SET {columna} = REPLACE(
                    {columna}, ?, ?
                )
                WHERE {columna} LIKE ?
                """,
                (
                    ruta_antigua,
                    ruta_nueva,
                    ruta_antigua + "%"
                )
            )

            cantidad = cursor.rowcount

            total_actualizaciones += cantidad

            print(
                f"{tabla}.{columna}: "
                f"{cantidad} referencias actualizadas"
            )

    conexion.commit()

    integridad = conexion.execute(
        "PRAGMA integrity_check"
    ).fetchone()[0]

    assert integridad == "ok"

finally:

    conexion.close()

print("Referencias actualizadas:",
      total_actualizaciones)

print("Integridad SQLite: CORRECTA")

# Configuración de MLFLOW restaurado
print("\n" + "=" * 65)
print("4. CONFIGURACIÓN DE MLFLOW RESTAURADO")
print("=" * 65)

mlflow.set_tracking_uri(
    f"sqlite:///{ruta_db.resolve()}"
)

cliente_restaurado = MlflowClient()

experimento = mlflow.get_experiment_by_name(
    "RAMA_PM25_Actividad5"
)

assert experimento is not None

assert experimento.artifact_location.startswith(
    ruta_nueva
)

print("Experimento:", experimento.name)
print("ID:", experimento.experiment_id)
print("Ruta de artefactos:", experimento.artifact_location)

# Identificación de modelos definitivos
print("\n" + "=" * 65)
print("5. IDENTIFICACIÓN DE MODELOS")
print("=" * 65)

modelos_esperados = {
    "Ridge Regression": {
        "model_id":
            "m-c6696e03f8294f8aa3075a56415f6b41",
        "run_id":
            "73339b012c7640e6bb99744dc3018c2d"
    },
    "Random Forest Regressor": {
        "model_id":
            "m-8de0647dad4c468d8e1308d0f261f528",
        "run_id":
            "3c164f2beea644eba2f4068cb3b7beb6"
    }
}

modelos_recuperados = {}
resultados_recuperacion = []

for nombre, identificadores in modelos_esperados.items():

    print("\n" + "-" * 65)
    print("Modelo:", nombre)
    print("-" * 65)

    model_id = identificadores["model_id"]
    run_id = identificadores["run_id"]

    carpeta_modelo = (
        ruta_artefactos
        / "models"
        / model_id
        / "artifacts"
    )

    ruta_mlmodel = carpeta_modelo / "MLmodel"

    assert ruta_mlmodel.exists(), (
        f"No se encontró MLmodel de {nombre}"
    )

    # Leer la configuración real del modelo.
    with open(
        ruta_mlmodel,
        "r",
        encoding="utf-8"
    ) as archivo:

        metadatos = yaml.safe_load(archivo)

    configuracion = metadatos["flavors"]["sklearn"]

    formato = configuracion["serialization_format"]
    archivo_modelo = configuracion["pickled_model"]

    ruta_modelo = carpeta_modelo / archivo_modelo

    assert ruta_modelo.exists(), (
        f"No se encontró {archivo_modelo} de {nombre}"
    )

    print("Formato:", formato)
    print("Archivo:", archivo_modelo)
    print("Tamaño:", ruta_modelo.stat().st_size, "bytes")

    # Verificar el registro de MLflow.
    registro = cliente_restaurado.get_logged_model(
        model_id
    )

    assert registro.status == "READY"

    assert registro.source_run_id == run_id

    assert registro.artifact_location.startswith(
        ruta_nueva
    )

    # Recuperar el modelo desde MLflow.
    uri_modelo = f"models:/{model_id}"

    modelo_cargado = mlflow.sklearn.load_model(
        uri_modelo
    )

    assert hasattr(modelo_cargado, "predict")

    modelos_recuperados[nombre] = modelo_cargado

    resultados_recuperacion.append({
        "Modelo": nombre,
        "Formato": formato,
        "Archivo": archivo_modelo,
        "Estado": registro.status,
        "Run_ID": run_id,
        "Model_ID": model_id,
        "Recuperacion": "CORRECTA"
    })

    print("Estado:", registro.status)
    print("Recuperación: CORRECTA")

# Verificación de predicciones
print("\n" + "=" * 65)
print("6. VERIFICACIÓN DE PREDICCIONES")
print("=" * 65)

# Los ejemplos de entrada fueron almacenados
# cuando se registraron los modelos en MLflow.

for nombre, identificadores in modelos_esperados.items():

    model_id = identificadores["model_id"]

    carpeta_modelo = (
        ruta_artefactos
        / "models"
        / model_id
        / "artifacts"
    )

    ruta_ejemplo = (
        carpeta_modelo / "input_example.json"
    )

    assert ruta_ejemplo.exists(), (
        f"No se encontró ejemplo de {nombre}"
    )

    with open(
        ruta_ejemplo,
        "r",
        encoding="utf-8"
    ) as archivo:

        contenido = json.load(archivo)

    # MLflow almacena los ejemplos de DataFrame
    # utilizando una representación tipo split.
    if "dataframe_split" in contenido:
        contenido = contenido["dataframe_split"]

    assert "columns" in contenido
    assert "data" in contenido

    X_ejemplo = pd.DataFrame(
        contenido["data"],
        columns=contenido["columns"]
    )

    # Recuperar los tipos esperados desde la firma.
    ruta_mlmodel = carpeta_modelo / "MLmodel"

    with open(
        ruta_mlmodel,
        "r",
        encoding="utf-8"
    ) as archivo:

        metadatos = yaml.safe_load(archivo)

    firma_entradas = json.loads(
        metadatos["signature"]["inputs"]
    )

    for entrada in firma_entradas:

        columna = entrada["name"]

        if entrada["type"] == "integer":
            X_ejemplo[columna] = (
                X_ejemplo[columna].astype("int32")
            )

        elif entrada["type"] == "double":
            X_ejemplo[columna] = (
                X_ejemplo[columna].astype("float64")
            )

    predicciones_recuperadas = (
        modelos_recuperados[nombre].predict(
            X_ejemplo
        )
    )

    assert len(predicciones_recuperadas) == len(
        X_ejemplo
    )

    assert np.isfinite(
        predicciones_recuperadas
    ).all()

    print("\nModelo:", nombre)
    print("Registros evaluados:", len(X_ejemplo))
    print("Predicciones generadas: CORRECTAS")
    print(
        "Primeras predicciones:",
        np.round(
            predicciones_recuperadas[:3],
            4
        ).tolist()
    )

# Tabla de resultados
print("\n" + "=" * 65)
print("7. RESUMEN DE MODELOS RECUPERADOS")
print("=" * 65)

df_restauracion_modelos = pd.DataFrame(
    resultados_recuperacion
)

display(df_restauracion_modelos)

# Verificación final
print("\n" + "=" * 65)
print("8. RESULTADO FINAL DE LA RECUPERACIÓN")
print("=" * 65)

assert len(modelos_recuperados) == 2

assert (
    df_restauracion_modelos["Estado"] == "READY"
).all()

print("Modelos recuperados:", len(modelos_recuperados))
print("Ridge Regression: CORRECTO")
print("Random Forest Regressor: CORRECTO")
print("Predicciones: VERIFICADAS")
print("Base restaurada: CORRECTA")
print("No se realizó ningún reentrenamiento.")

print("\nCelda 4 completada satisfactoriamente.")


1. CONFIGURACIÓN DE LA RESTAURACIÓN
MLflow: 3.17.1
Base restaurada: /content/mlflow_restaurado/mlflow.db
Artefactos: /content/mlflow_restaurado/artefactos
Ruta esperada: file:///content/mlflow_restaurado/artefactos

2. VERIFICACIÓN DE DEPENDENCIAS
skops: DISPONIBLE
cloudpickle: 3.1.2

3. VERIFICACIÓN DE REFERENCIAS SQLITE
Copia de seguridad: EXISTENTE
experiments.artifact_location: 0 referencias actualizadas
runs.artifact_uri: 0 referencias actualizadas
logged_models.artifact_location: 0 referencias actualizadas
Referencias actualizadas: 0
Integridad SQLite: CORRECTA

4. CONFIGURACIÓN DE MLFLOW RESTAURADO
Experimento: RAMA_PM25_Actividad5
ID: 1
Ruta de artefactos: file:///content/mlflow_restaurado/artefactos

5. IDENTIFICACIÓN DE MODELOS

-----------------------------------------------------------------
Modelo: Ridge Regression
-----------------------------------------------------------------
Formato: skops
Archivo: model.skops
Tamaño: 27100 bytes
Estado: READY
Recuperación: CORRECTA



,Modelo,Formato,Archivo,Estado,Run_ID,Model_ID,Recuperacion
0,Ridge Regression,skops,model.skops,READY,73339b012c7640e6bb99744dc3018c2d,m-c6696e03f8294f8aa3075a56415f6b41,CORRECTA
1,Random Forest Regressor,cloudpickle,model.pkl,READY,3c164f2beea644eba2f4068cb3b7beb6,m-8de0647dad4c468d8e1308d0f261f528,CORRECTA



8. RESULTADO FINAL DE LA RECUPERACIÓN
Modelos recuperados: 2
Ridge Regression: CORRECTO
Random Forest Regressor: CORRECTO
Predicciones: VERIFICADAS
Base restaurada: CORRECTA
No se realizó ningún reentrenamiento.

Celda 4 completada satisfactoriamente.


In [10]:
# TAREA 5
# Restauración de MLFLOW
# 5. Verificación integral de resultados y artefactos

import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow

from mlflow.tracking import MlflowClient
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Configuración del experimento restaurado
print("=" * 65)
print("1. CONFIGURACIÓN DE LA VERIFICACIÓN FINAL")
print("=" * 65)

ruta_db = Path(
    "/content/mlflow_restaurado/mlflow.db"
)

assert ruta_db.exists(), (
    "No se encontró la base de datos restaurada."
)

mlflow.set_tracking_uri(
    f"sqlite:///{ruta_db.resolve()}"
)

cliente_final = MlflowClient()

experimento = mlflow.get_experiment_by_name(
    "RAMA_PM25_Actividad5"
)

assert experimento is not None

print("Experimento:", experimento.name)
print("MLflow:", mlflow.__version__)
print("Base restaurada:", ruta_db)

runs_definitivos = {
    "Ridge Regression":
        "73339b012c7640e6bb99744dc3018c2d",
    "Random Forest Regressor":
        "3c164f2beea644eba2f4068cb3b7beb6"
}

# Verificación de ejecuciones y métricas
print("\n" + "=" * 65)
print("2. VERIFICACIÓN DE EJECUCIONES Y MÉTRICAS")
print("=" * 65)

metricas_requeridas = [
    "CV_MAE",
    "CV_RMSE",
    "CV_R2",
    "Test_MAE",
    "Test_RMSE",
    "Test_R2"
]

referencias = {
    "Ridge Regression": {
        "CV_MAE": 4.2367,
        "CV_RMSE": 5.7668,
        "CV_R2": 0.5405,
        "Test_MAE": 3.6493,
        "Test_RMSE": 4.9894,
        "Test_R2": 0.4861
    },
    "Random Forest Regressor": {
        "CV_MAE": 4.3197,
        "CV_RMSE": 5.8419,
        "CV_R2": 0.5264,
        "Test_MAE": 3.6743,
        "Test_RMSE": 5.1004,
        "Test_R2": 0.4629
    }
}

registros = []

for nombre, run_id in runs_definitivos.items():

    registro = cliente_final.get_run(run_id)

    assert registro.info.status == "FINISHED"

    assert all(
        metrica in registro.data.metrics
        for metrica in metricas_requeridas
    )

    assert len(registro.data.params) > 0

    fila = {
        "Modelo": nombre,
        "Run_ID": run_id,
        "Estado": registro.info.status
    }

    for metrica in metricas_requeridas:

        valor = float(
            registro.data.metrics[metrica]
        )

        esperado = referencias[nombre][metrica]

        assert abs(valor - esperado) <= 0.0001, (
            f"Diferencia en {nombre}: {metrica}"
        )

        fila[metrica] = valor

    registros.append(fila)

    print("Modelo:", nombre)
    print("Estado:", registro.info.status)
    print("Parámetros:", len(registro.data.params))
    print("Métricas verificadas:", len(metricas_requeridas))

df_metricas_restauradas = pd.DataFrame(registros)

display(
    df_metricas_restauradas[
        [
            "Modelo",
            "CV_MAE",
            "CV_RMSE",
            "CV_R2",
            "Test_MAE",
            "Test_RMSE",
            "Test_R2"
        ]
    ].round(4)
)

# Verificación de artefactos de evaluación
print("\n" + "=" * 65)
print("3. VERIFICACIÓN DE ARTEFACTOS RESTAURADOS")
print("=" * 65)

resultados_evaluacion = []

for nombre, run_id in runs_definitivos.items():

    print("-" * 65)
    print("Modelo:", nombre)
    print("-" * 65)

    artefactos = cliente_final.list_artifacts(
        run_id,
        path="evaluacion"
    )

    rutas = [archivo.path for archivo in artefactos]

    archivos_requeridos = [
        "evaluacion/predicciones_prueba.csv",
        "evaluacion/resumen_modelo.json"
    ]

    for archivo in archivos_requeridos:

        assert archivo in rutas, (
            f"Falta el artefacto {archivo} de {nombre}"
        )

    # Descargar artefactos desde la copia restaurada.
    with tempfile.TemporaryDirectory() as carpeta:

        ruta_csv = cliente_final.download_artifacts(
            run_id,
            "evaluacion/predicciones_prueba.csv",
            dst_path=carpeta
        )

        ruta_json = cliente_final.download_artifacts(
            run_id,
            "evaluacion/resumen_modelo.json",
            dst_path=carpeta
        )

        df_pred = pd.read_csv(ruta_csv)

        with open(
            ruta_json,
            "r",
            encoding="utf-8"
        ) as archivo:

            resumen = json.load(archivo)

        assert len(df_pred) == 609

        assert all(
            columna in df_pred.columns
            for columna in [
                "PM25_real",
                "PM25_predicho"
            ]
        )

        assert resumen["modelo"] == nombre

        y_real = df_pred["PM25_real"].to_numpy()
        y_estimado = df_pred["PM25_predicho"].to_numpy()

        assert np.isfinite(y_real).all()
        assert np.isfinite(y_estimado).all()

        mae = mean_absolute_error(
            y_real,
            y_estimado
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_real,
                y_estimado
            )
        )

        r2 = r2_score(
            y_real,
            y_estimado
        )

        metricas_registradas = cliente_final.get_run(
            run_id
        ).data.metrics

        np.testing.assert_allclose(
            [
                mae,
                rmse,
                r2
            ],
            [
                metricas_registradas["Test_MAE"],
                metricas_registradas["Test_RMSE"],
                metricas_registradas["Test_R2"]
            ],
            rtol=1e-10,
            atol=1e-10
        )

        resultados_evaluacion.append({
            "Modelo": nombre,
            "Registros": len(df_pred),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "Artefactos": "VERIFICADOS"
        })

        print("Predicciones:", len(df_pred))
        print("Archivo JSON: CORRECTO")
        print("Métricas recalculadas: COINCIDEN")

# Tabla final de reproducibilidad
print("\n" + "=" * 65)
print("4. RESULTADOS DE LA VERIFICACIÓN")
print("=" * 65)

df_resultados_restauracion = pd.DataFrame(
    resultados_evaluacion
)

display(
    df_resultados_restauracion.round(4)
)

# Resultado final
print("\n" + "=" * 65)
print("5. RESULTADO FINAL DE LA RESTAURACIÓN")
print("=" * 65)

assert len(df_resultados_restauracion) == 2

assert (
    df_resultados_restauracion["Artefactos"]
    == "VERIFICADOS"
).all()

print("Modelos restaurados: 2")
print("Ejecuciones definitivas: FINISHED")
print("Métricas originales: VERIFICADAS")
print("Artefactos restaurados: VERIFICADOS")
print("Predicciones: 609 por modelo")
print("Métricas recalculadas: COINCIDEN")

print("\nRESTAURACIÓN FUNCIONAL DE MLFLOW: SATISFACTORIA")
print("No se realizó ningún reentrenamiento.")

1. CONFIGURACIÓN DE LA VERIFICACIÓN FINAL
Experimento: RAMA_PM25_Actividad5
MLflow: 3.17.1
Base restaurada: /content/mlflow_restaurado/mlflow.db

2. VERIFICACIÓN DE EJECUCIONES Y MÉTRICAS
Modelo: Ridge Regression
Estado: FINISHED
Parámetros: 7
Métricas verificadas: 6
Modelo: Random Forest Regressor
Estado: FINISHED
Parámetros: 9
Métricas verificadas: 6


,Modelo,CV_MAE,CV_RMSE,CV_R2,Test_MAE,Test_RMSE,Test_R2
0,Ridge Regression,4.2367,5.7668,0.5405,3.6493,4.9894,0.4861
1,Random Forest Regressor,4.3197,5.8419,0.5264,3.6743,5.1004,0.4629



3. VERIFICACIÓN DE ARTEFACTOS RESTAURADOS
-----------------------------------------------------------------
Modelo: Ridge Regression
-----------------------------------------------------------------


Predicciones: 609
Archivo JSON: CORRECTO
Métricas recalculadas: COINCIDEN
-----------------------------------------------------------------
Modelo: Random Forest Regressor
-----------------------------------------------------------------


Predicciones: 609
Archivo JSON: CORRECTO
Métricas recalculadas: COINCIDEN

4. RESULTADOS DE LA VERIFICACIÓN


,Modelo,Registros,MAE,RMSE,R2,Artefactos
0,Ridge Regression,609,3.6493,4.9894,0.4861,VERIFICADOS
1,Random Forest Regressor,609,3.6743,5.1004,0.4629,VERIFICADOS



5. RESULTADO FINAL DE LA RESTAURACIÓN
Modelos restaurados: 2
Ejecuciones definitivas: FINISHED
Métricas originales: VERIFICADAS
Artefactos restaurados: VERIFICADOS
Predicciones: 609 por modelo
Métricas recalculadas: COINCIDEN

RESTAURACIÓN FUNCIONAL DE MLFLOW: SATISFACTORIA
No se realizó ningún reentrenamiento.
